# Visualization 7: Consumption Trends Analysis
## USDA Food Consumption Data (Table 5): 1977-2018

**Purpose:** Analyze protein food consumption trends over time, focusing on cured meat (RTE products)

**Data Source:** `table-5-US-food-group-intakes-by-food-source.csv`

**Key Questions:**
1. How has cured meat consumption changed over time?
2. What are the most consumed protein categories in 2017-2018?
3. When did consumption peak and why did it decline?

**Output:** Cleaned consumption data for integration with contamination analysis

---
## Section 1: Setup and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from datetime import datetime

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

---
## Section 2: Load Consumption Data

In [ ]:
# Load the consumption data CSV
csv_path = 'table-5-US-food-group-intakes-by-food-source.csv'

try:
    df_full = pd.read_csv(csv_path)
    print(f"✓ Data loaded successfully: {len(df_full):,} total records")
    print(f"\nColumns: {list(df_full.columns)}")
    print(f"\nData shape: {df_full.shape}")
except FileNotFoundError:
    print(f"❌ ERROR: File not found at {csv_path}")
    print("Please ensure the CSV file is in the current directory.")

In [ ]:
# Preview the data structure
print("First 5 rows of raw data:")
df_full.head()

In [ ]:
# Filter for protein foods only
df_protein = df_full[
    df_full['Food group'].str.contains('Protein foods', case=False, na=False)
].copy()

# Filter for Food at Home (FAH) only - closest to RTE retail purchases
df_protein = df_protein[df_protein['Food source'] == 'FAH']

# Filter for US average (not demographic subgroups)
df_protein = df_protein[
    df_protein['Demographics'] == 'US consumers aged 2 and above'
]

# Filter for Mean values (not Standard Error)
df_protein = df_protein[
    df_protein['Survey years:Variable'].str.contains('Mean', na=False)
]

print(f"✓ Filtered to protein foods: {len(df_protein):,} records")
print(f"\nUnique protein categories: {df_protein['Food group'].nunique()}")
print(f"Unique survey periods: {df_protein['Survey years:Variable'].nunique()}")

In [ ]:
# Show available protein categories
print("Available protein food categories:")
for cat in sorted(df_protein['Food group'].unique()):
    print(f"  - {cat}")

---
## Section 3: Data Cleaning and Transformation

In [ ]:
def clean_consumption_data(df):
    """
    Clean and transform consumption data
    
    Returns:
        clean_df: Cleaned dataframe with parsed years and calculated values
    """
    df_clean = df.copy()
    
    # Extract year from 'Survey years:Variable' (e.g., "2017-2018-Mean" -> "2017-2018")
    df_clean['year_range'] = df_clean['Survey years:Variable'].str.replace('-Mean', '')
    
    # Create year for plotting (use start year)
    def parse_year(year_str):
        if '-' in year_str:
            years = year_str.split('-')
            start_year = int(years[0])
            return start_year
        return None
    
    df_clean['year'] = df_clean['year_range'].apply(parse_year)
    
    # Ensure Value is numeric
    df_clean['value_oz_per_day'] = pd.to_numeric(df_clean['Value'], errors='coerce')
    
    # Calculate lbs per year (oz/day * 365 days / 16 oz per lb)
    df_clean['value_lbs_per_year'] = df_clean['value_oz_per_day'] * 365 / 16
    
    # Remove rows with missing values
    df_clean = df_clean.dropna(subset=['year', 'value_oz_per_day'])
    
    return df_clean

df_protein_clean = clean_consumption_data(df_protein)

print("✓ Data cleaned successfully")
print(f"\nRecords after cleaning: {len(df_protein_clean):,}")
print(f"Year range: {df_protein_clean['year'].min()} - {df_protein_clean['year'].max()}")

In [ ]:
# Preview cleaned data
print("Cleaned data sample (Cured meat):")
df_protein_clean[
    df_protein_clean['Food group'] == 'Protein foods, cured meat'
][['Food group', 'year', 'value_oz_per_day', 'value_lbs_per_year']].sort_values('year')

---
## Section 4: Visualization 1 - Cured Meat Consumption Over Time

In [ ]:
# Filter for cured meat only
cured_meat = df_protein_clean[
    df_protein_clean['Food group'] == 'Protein foods, cured meat'
].copy()
cured_meat = cured_meat.sort_values('year')

print(f"Cured meat records: {len(cured_meat)}")
print(f"\nCured meat consumption data:")
print(cured_meat[['year', 'value_oz_per_day', 'value_lbs_per_year']])

In [ ]:
# Create dual-axis line chart
fig, ax1 = plt.subplots(figsize=(16, 7))

# Plot oz/day on left axis
color1 = '#d62728'
ax1.plot(cured_meat['year'], cured_meat['value_oz_per_day'],
         marker='o', linewidth=2.5, markersize=10, color=color1, label='Oz/Day')
ax1.set_xlabel('Year', fontsize=13, fontweight='bold')
ax1.set_ylabel('Ounces per Day', fontsize=13, fontweight='bold', color=color1)
ax1.tick_params(axis='y', labelcolor=color1, labelsize=11)
ax1.tick_params(axis='x', labelsize=11)
ax1.grid(True, alpha=0.3)

# Plot lbs/year on right axis
ax2 = ax1.twinx()
color2 = '#1f77b4'
ax2.plot(cured_meat['year'], cured_meat['value_lbs_per_year'],
         marker='s', linewidth=2.5, markersize=10, color=color2,
         linestyle='--', label='Lbs/Year')
ax2.set_ylabel('Pounds per Year', fontsize=13, fontweight='bold', color=color2)
ax2.tick_params(axis='y', labelcolor=color2, labelsize=11)

# Find and annotate peak
peak_idx = cured_meat['value_oz_per_day'].idxmax()
peak_year = cured_meat.loc[peak_idx, 'year']
peak_value_oz = cured_meat.loc[peak_idx, 'value_oz_per_day']
peak_value_lbs = cured_meat.loc[peak_idx, 'value_lbs_per_year']

ax1.annotate(f'PEAK: {peak_value_oz:.2f} oz/day\n({peak_value_lbs:.1f} lbs/year)\n{int(peak_year)}',
             xy=(peak_year, peak_value_oz),
             xytext=(peak_year - 8, peak_value_oz + 0.05),
             arrowprops=dict(arrowstyle='->', color='red', lw=2.5),
             fontsize=12, fontweight='bold', color='red',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

# Annotate 2011 (major outbreak year)
if 2011 in cured_meat['year'].values:
    ax1.axvline(x=2011, color='orange', linestyle=':', linewidth=2.5, alpha=0.7)
    ax1.text(2011, ax1.get_ylim()[1] * 0.95, '2011\nMajor Listeria\nOutbreaks',
             ha='center', fontsize=11, color='orange', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

# Annotate 2015 (WHO warning)
if 2015 in cured_meat['year'].values:
    ax1.axvline(x=2015, color='purple', linestyle=':', linewidth=2.5, alpha=0.7)
    ax1.text(2015, ax1.get_ylim()[1] * 0.85, '2015\nWHO Cancer\nWarning',
             ha='center', fontsize=11, color='purple', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

# Title and legend
plt.title('Cured Meat (RTE) Consumption Trends: 1977-2018\n15% Decline After 2009 Peak',
          fontsize=16, fontweight='bold', pad=20)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=12, framealpha=0.9)

plt.tight_layout()
plt.savefig('consumption_cured_meat_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: consumption_cured_meat_trend.png")

In [ ]:
# Calculate key statistics
recent_year = 2017
recent_value_oz = cured_meat[cured_meat['year'] == recent_year]['value_oz_per_day'].values[0]
recent_value_lbs = cured_meat[cured_meat['year'] == recent_year]['value_lbs_per_year'].values[0]
decline_pct = ((recent_value_oz - peak_value_oz) / peak_value_oz) * 100

print("="*80)
print("KEY FINDINGS: Cured Meat Consumption")
print("="*80)
print(f"Peak Year: {int(peak_year)}")
print(f"Peak Consumption: {peak_value_oz:.2f} oz/day ({peak_value_lbs:.1f} lbs/year)")
print(f"\nRecent (2017): {recent_value_oz:.2f} oz/day ({recent_value_lbs:.1f} lbs/year)")
print(f"\nDecline: {abs(decline_pct):.1f}% from peak to 2017")
print(f"Absolute change: {abs(recent_value_lbs - peak_value_lbs):.1f} lbs/year decrease")
print("="*80)

---
## Section 5: Visualization 2 - Protein Category Comparison (2017-2018)

In [ ]:
# Filter for 2017 data
df_2017 = df_protein_clean[df_protein_clean['year'] == 2017].copy()

# Get unique food groups and their values
categories_2017 = df_2017.groupby('Food group')['value_oz_per_day'].first().reset_index()

# Keep main categories only
main_categories = [
    'Protein foods, total',
    'Protein foods, meats, poultry, and fish',
    'Protein foods, meats (beef, veal, pork, lamb, game)',
    'Protein foods, poultry',
    'Protein foods, cured meat',
    'Protein foods, nuts and seeds',
    'Protein foods, eggs',
    'Protein foods, low Omega-3 fatty fish',
    'Protein foods, high Omega-3 fatty fish',
    'Protein foods, soy products',
    'Protein foods, organ meats'
]

categories_2017 = categories_2017[
    categories_2017['Food group'].isin(main_categories)
].copy()
categories_2017['value_lbs_per_year'] = categories_2017['value_oz_per_day'] * 365 / 16
categories_2017 = categories_2017.sort_values('value_oz_per_day', ascending=True)

print(f"Categories for 2017-2018 comparison: {len(categories_2017)}")

In [ ]:
# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(14, 9))

# Create bars
bars = ax.barh(categories_2017['Food group'], categories_2017['value_oz_per_day'],
               color='#2ca02c', alpha=0.7, edgecolor='black', linewidth=1.5)

# Highlight cured meat
cured_idx = categories_2017[
    categories_2017['Food group'] == 'Protein foods, cured meat'
].index[0]
cured_pos = list(categories_2017.index).index(cured_idx)
bars[cured_pos].set_color('#d62728')
bars[cured_pos].set_alpha(1.0)
bars[cured_pos].set_linewidth(2.5)

# Add value labels
for i, (idx, row) in enumerate(categories_2017.iterrows()):
    oz_val = row['value_oz_per_day']
    lbs_val = row['value_lbs_per_year']
    ax.text(oz_val + 0.08, i, f'{oz_val:.2f} oz/day\n({lbs_val:.1f} lbs/yr)',
            va='center', fontsize=10, fontweight='bold')

# Labels and title
ax.set_xlabel('Ounces per Day per Person', fontsize=13, fontweight='bold')
ax.set_ylabel('Protein Category', fontsize=13, fontweight='bold')
ax.set_title('Protein Food Consumption by Category (2017-2018)\nCured Meat Highlighted in Red',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('consumption_category_comparison_2017.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: consumption_category_comparison_2017.png")

In [ ]:
# Display ranking table
print("\nProtein Consumption Ranking (2017-2018):")
print("="*80)
ranking = categories_2017.sort_values('value_oz_per_day', ascending=False)
for i, (idx, row) in enumerate(ranking.iterrows(), 1):
    cat = row['Food group'].replace('Protein foods, ', '')
    oz = row['value_oz_per_day']
    lbs = row['value_lbs_per_year']
    print(f"{i:2d}. {cat:50s} {oz:5.2f} oz/day ({lbs:5.1f} lbs/year)")
print("="*80)

---
## Section 6: Visualization 3 - Temporal Heatmap

In [ ]:
# Select key categories for heatmap
key_categories = [
    'Protein foods, cured meat',
    'Protein foods, poultry',
    'Protein foods, meats (beef, veal, pork, lamb, game)',
    'Protein foods, eggs'
]

df_heatmap = df_protein_clean[
    df_protein_clean['Food group'].isin(key_categories)
].copy()

# Pivot to create matrix
pivot = df_heatmap.pivot_table(
    values='value_oz_per_day',
    index='Food group',
    columns='year',
    aggfunc='first'
)

# Shorten labels
pivot.index = pivot.index.str.replace('Protein foods, ', '')

print(f"Heatmap data shape: {pivot.shape}")

In [ ]:
# Create heatmap
fig, ax = plt.subplots(figsize=(16, 7))

sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=1.5, linecolor='black',
            cbar_kws={'label': 'Ounces per Day'},
            ax=ax, annot_kws={'fontsize': 10})

ax.set_title('Protein Consumption Trends: Key Categories (1977-2018)\nColor Intensity = Higher Consumption',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=13, fontweight='bold')
ax.set_ylabel('Protein Category', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('consumption_heatmap_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: consumption_heatmap_trends.png")

---
## Section 7: Export Cleaned Data for Integration

In [ ]:
# Save cleaned consumption data
output_csv = 'processed_consumption_data.csv'
df_protein_clean.to_csv(output_csv, index=False)
print(f"✓ Cleaned consumption data exported: {output_csv}")
print(f"  Records: {len(df_protein_clean):,}")
print(f"  Columns: {list(df_protein_clean.columns)}")

In [ ]:
# Create summary statistics JSON
summary_stats = {
    'analysis_date': datetime.now().strftime('%Y-%m-%d'),
    'data_source': 'table-5-US-food-group-intakes-by-food-source.csv',
    'cured_meat': {
        'peak_year': int(peak_year),
        'peak_oz_per_day': float(peak_value_oz),
        'peak_lbs_per_year': float(peak_value_lbs),
        'recent_year': 2017,
        'recent_oz_per_day': float(recent_value_oz),
        'recent_lbs_per_year': float(recent_value_lbs),
        'decline_percent': float(decline_pct)
    },
    'categories_2017': {
        'poultry_oz_per_day': float(categories_2017[
            categories_2017['Food group'] == 'Protein foods, poultry'
        ]['value_oz_per_day'].values[0]),
        'meats_oz_per_day': float(categories_2017[
            categories_2017['Food group'] == 'Protein foods, meats (beef, veal, pork, lamb, game)'
        ]['value_oz_per_day'].values[0]),
        'cured_meat_oz_per_day': float(recent_value_oz)
    }
}

output_json = 'consumption_summary_stats.json'
with open(output_json, 'w') as f:
    json.dump(summary_stats, f, indent=2)

print(f"\n✓ Summary statistics exported: {output_json}")
print("\nSummary:")
print(json.dumps(summary_stats, indent=2))

---
## Section 8: Summary and Next Steps

In [ ]:
print("="*80)
print("ANALYSIS COMPLETE: Consumption Trends")
print("="*80)
print("\n📊 VISUALIZATIONS CREATED:")
print("   1. consumption_cured_meat_trend.png - Time series (1977-2018)")
print("   2. consumption_category_comparison_2017.png - Category ranking")
print("   3. consumption_heatmap_trends.png - Multi-category heatmap")
print("\n💾 DATA EXPORTED:")
print("   1. processed_consumption_data.csv - Clean data for integration")
print("   2. consumption_summary_stats.json - Key statistics")
print("\n🔍 KEY FINDINGS:")
print(f"   • Cured meat consumption PEAKED in {int(peak_year)} at {peak_value_lbs:.1f} lbs/year")
print(f"   • DECLINED {abs(decline_pct):.1f}% by 2017 to {recent_value_lbs:.1f} lbs/year")
print("   • Decline coincides with 2011 outbreaks and 2015 WHO warning")
print("\n➡️  NEXT STEP:")
print("   Run visualization_8_consumption_vs_contamination.ipynb")
print("   to combine this data with contamination analysis")
print("="*80)